In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import sys
sys.path.insert(0, './src')
from iot_producer import IoTProducer
from digital_twin import DigitalTwinConsumer

WINDOW_SIZE = 500
STEP = 100
BROKER = "kafka:9092"
TOPIC = "telemetry"

os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/logs', exist_ok=True)

In [ ]:
# Load baseline dataset and send to Kafka
df = pd.read_csv('data/baseline_dataset.csv')
signal = df['sensor_value'].values
print(f"Dataset loaded: {len(signal)} samples")

# Initialize producer and send data to Kafka
producer = IoTProducer(broker=BROKER, topic=TOPIC)
print("sending data")
for i, value in enumerate(signal):
    producer.send_data(timestamp=i, value=value)
    if (i + 1) % 1000 == 0:
        print(f"Sent {i + 1}/{len(signal)} samples")
producer.flush()

# Consume data from Kafka with metrics analysis in digital twin
consumer = DigitalTwinConsumer(broker=BROKER, topic=TOPIC, group="entropy-analysis-v2")
entropy_history = []

# Collect metrics directly from digital twin
max_collect_attempts = 100
attempts = 0
while attempts < max_collect_attempts:
    metrics = consumer.analyse_metrics(window_size=WINDOW_SIZE, timeout_ms=5000)
    if metrics is not None:
        entropy_history.append(metrics)
        #print(f"  → Analizzata finestra {len(entropy_history)}: Shannon={metrics['shannon']:.4f}")
    else:
        if len(entropy_history) > 0:
            break
    attempts += 1

print(f"Analisi completata: {len(entropy_history)} finestre elaborate")

consumer.close()

df_h = pd.DataFrame(entropy_history) if entropy_history else pd.DataFrame()
print(f"\nCalcolo completato per {len(df_h)} finestre temporali.")

In [ ]:
if len(df_h) == 0:
    print("Errore: nessun dato elaborato")
    baseline_stats = {
        'shannon_mean': 0,
        'shannon_std': 0,
        'sample_en_mean': 0,
        'sample_en_std': 0
    }
else:
    baseline_stats = {
        'shannon_mean': df_h['shannon'].mean(),
        'shannon_std': df_h['shannon'].std(),
        'sample_en_mean': df_h['sample_en'].mean(),
        'sample_en_std': df_h['sample_en'].std()
    }

with open('results/logs/baseline_metrics.json', 'w') as f:
    json.dump(baseline_stats, f)

print(f"Baseline stabilita: Shannon Mean = {baseline_stats['shannon_mean']:.4f}")

In [ ]:
if len(df_h) == 0:
    print("⚠️  Nessun dato da visualizzare")
else:
    plt.figure(figsize=(12, 5))

    # Plot 1: Serie temporale dell'entropia
    plt.subplot(1, 2, 1)
    plt.plot(df_h['shannon'], color='blue', label='Shannon H')
    plt.axhline(baseline_stats['shannon_mean'], color='red', linestyle='--', label='Mean')
    plt.title("H(k) Stability - Baseline")
    plt.xlabel("Window Index")
    plt.ylabel("Entropy")
    plt.legend()

    # Plot 2: Distribuzione (Istogramma)
    plt.subplot(1, 2, 2)
    sns.histplot(df_h['shannon'], kde=True, color='green')
    plt.title("Entropy Distribution")

    plt.tight_layout()
    plt.savefig('results/figures/baseline_stability.png')
    print("✅ Grafico salvato in results/figures/baseline_stability.png")
    plt.show()